In [ ]:
# 🛠️ Setup: install deps + fetch the shared helper on Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q transformer_lens datasets tokenizers plotly
    !wget -q https://raw.githubusercontent.com/barmag/fanous-llm-lens/main/notebooks/education/tiny.py
import tiny
import torch
import plotly.graph_objects as go

torch.manual_seed(42)
print("device:", tiny.device())

<div dir="rtl">

# المرحلة ٢أ: أول كتلة انتباه — لما الكلمة تقدر تبص لجيرانها

في المرحلة ١ج بنينا نموذج "تضمينات بس" (embeddings only): كل وحدة فرعية (subword) ليها متجه (vector) ثابت، والنموذج بيتنبأ بالكلمة اللي جاية من غير ما يبص لأي كلمة تانية في الجملة. ده نموذج **أعمى للسياق** (context-blind).

النهارده بنضيف حاجة واحدة بس: **كتلة انتباه واحدة برأس واحد** (one attention block, one head). الإضافة دي بتدي الموديل قدرة جديدة: الكلمة تقدر **تبص للكلمات اللي قبلها** وتستعمل المعلومة دي وهي بتتنبأ.

**حدود التجربة (مهم):** الموديل ده صغير جدًا واتدرب على بيانات قليلة، فالتوقعات هتبان ضعيفة — وده طبيعي. هدفنا مش نبني موديل شاطر؛ هدفنا **نشوف الرأس وهو بيتعلم يبص**.

</div>

<div dir="ltr">

# Stage 2a: The first attention block — when a token can look at its neighbours

Stage 1c gave us an *embeddings-only* model: every subword has a fixed vector, and the model predicts the next token **without looking at any other token**. That model is **context-blind**.

Today we add exactly one thing: **one attention block with one head**. That single addition unlocks a new capability — a token can **look at the tokens before it** and use that information when predicting.

**Limits (important):** this model is tiny and trained on little data, so its predictions will look weak. That's fine — the goal isn't a smart model, it's to *watch the head learn to point*.

</div>

<div dir="rtl">

## ١. البيانات · Data

هنستعمل نفس مصادر المرحلة ١ج: نصوص فصحى (MSA) من ويكيبيديا، ونصوص مصري (Masri) من تويتات. هندرّب مُجزّئ BPE صغير على الاتنين مع بعض، وبعدين نحوّل النص لأرقام (token ids). حجم القاموس (`VOCAB`) بييجي من المُجزّئ نفسه — عشان أرقام التوكنز ما تتعداش حجم الموديل.

</div>

In [ ]:
# 📦 Fetch MSA + Masri text and learn a small BPE vocabulary (same sources as Stage 1c)
from datasets import load_dataset
import re
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Knobs (the test harness shrinks these so the notebook runs in seconds):
MAX_CHARS = globals().get("MAX_CHARS", 500_000)
BPE_VOCAB = globals().get("BPE_VOCAB", 3000)

print("Fetching MSA from Wikipedia and Masri from Tweets...")
msa_stream = load_dataset("wikimedia/wikipedia", "20231101.ar", split="train", streaming=True)
tweets_ds = load_dataset("amgadhasan/arabic_tweets_dialects", split="train")
eg_tweets = tweets_ds.filter(lambda x: x["dialect"] == "EG")

def clean(text):
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[a-zA-Z0-9_@]+", "", text)
    return re.sub(r"[^\s\u0621-\u064A]", "", text)

def collect(rows, key, max_chars):
    out = ""
    for r in rows:
        out += clean(r[key]) + " "
        if len(out) >= max_chars:
            break
    return out[:max_chars]

msa_text = collect(msa_stream, "text", MAX_CHARS)
masri_text = collect(eg_tweets, "text", MAX_CHARS)

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=BPE_VOCAB, special_tokens=["[UNK]"])
tokenizer.train_from_iterator([msa_text, masri_text], trainer)

VOCAB = tokenizer.get_vocab_size()
id_to_token = {i: t for t, i in tokenizer.get_vocab().items()}
all_ids = tokenizer.encode(msa_text).ids + tokenizer.encode(masri_text).ids

print(f"vocab: {VOCAB} subwords  |  tokens: {len(all_ids)}")  # ← (n_ids,)

<div dir="rtl">

## ٢. نبني الموديل وندرّبه · Build & train

هنبني موديل بطبقة واحدة (`n_layers=1`) ورأس انتباه واحد (`n_heads=1`)، وندرّبه يتنبأ بالـ subword اللي جاية. الفرق الوحيد عن موديل المرحلة ١ج هو **كتلة الانتباه** دي. لاحظ الخسارة (loss) وهي بتنزل.

</div>

In [ ]:
N_CTX = globals().get("N_CTX", 64)
N_EPOCHS = globals().get("N_EPOCHS", 200)

batches = tiny.make_natural_batches(all_ids, n_ctx=N_CTX)
print("batches:", tuple(batches.shape))  # ← (N, n_ctx)

model = tiny.make_tiny_model(n_layers=1, n_heads=1, d_vocab=VOCAB, n_ctx=N_CTX)
losses = tiny.train(model, batches, n_epochs=N_EPOCHS)
print("loss:", round(losses[0], 3), "->", round(losses[-1], 3))  # ← scalar

<div dir="rtl">

## ٣. بإيدينا · By hand

الانتباه مش صندوق أسود. هنحسب نمط الانتباه (attention pattern) لجملة واحدة **بإيدينا** عن طريق ضرب مصفوفتين: الاستعلام (Query, `q`) في المفتاح (Key, `k`)، وبعدين softmax. وبعدها نثبت إن النتيجة بتطابق اللي الموديل بيطلعه من جوّاه بالظبط — يعني الفرق صفر.

</div>

In [ ]:
prompt = "القطة بتاكل السمك"
enc = tokenizer.encode(prompt)
ids = torch.tensor([enc.ids]).to(tiny.device())
str_toks = enc.tokens
print("tokens:", str_toks)  # ← (seq,)

_, cache = model.run_with_cache(ids)
q = cache["q", 0][0, :, 0, :]                 # ← (seq, d_head)
k = cache["k", 0][0, :, 0, :]                 # ← (seq, d_head)
scores = (q @ k.T) / (q.shape[-1] ** 0.5)     # ← (seq, seq)
mask = torch.triu(torch.ones_like(scores), diagonal=1).bool()
by_hand = torch.softmax(scores.masked_fill(mask, float("-inf")), dim=-1)

from_cache = cache["pattern", 0][0, 0]        # ← (seq, seq)
print("max abs diff (by-hand vs model):", float((by_hand - from_cache).abs().max()))  # ← ~0

<div dir="rtl">

## ٤. الخريطة الحرارية · The heatmap

الرقم فوق قريب جدًا من الصفر، يبقى الصورة اللي تحت **هي** مصفوفة الانتباه نفسها — مش رسمة تقريبية ليها. كل صف بيقول: الـ subword ده وزّع انتباهه إزاي على الكلمات اللي قبله. (بنعرض من اليمين للشمال زي العربي.)

</div>

In [ ]:
def show_attention(model, prompt_ids, str_tokens, layer=0, head=0, title=""):
    _, cache = model.run_with_cache(prompt_ids)
    pattern = cache["pattern", layer][0, head].detach().cpu().numpy()  # ← (seq, seq)
    fig = go.Figure(go.Heatmap(z=pattern, x=str_tokens, y=str_tokens, colorscale="Blues"))
    fig.update_layout(title=title, xaxis=dict(side="top"))
    fig.update_xaxes(autorange="reversed")  # RTL: first Arabic token on the right
    fig.update_yaxes(autorange="reversed")
    return fig

show_attention(model, ids, str_toks, 0, 0, "Head 0 · القطة بتاكل السمك").show()

<div dir="rtl">

## ٥. قبل وبعد · Before / after

نموذج "التضمينات بس" بتاع المرحلة ١ج كان بيتنبأ بنفس التوزيع مهما كانت الكلمات اللي قبل — لأنه أعمى للسياق. دلوقتي، بعد ما ضفنا الانتباه، التوقع بيتغير حسب السياق. نطبع أعلى ٥ توقعات للـ subword الجاية بعد الجملة:

</div>

In [ ]:
last_logits = model(ids)[0, -1]                     # ← (vocab,)
probs = torch.softmax(last_logits, dim=-1)
topk = torch.topk(probs, 5)
print("Top-5 next subwords after the prompt:")
for p, i in zip(topk.values.tolist(), topk.indices.tolist()):
    print(f"  {id_to_token.get(i, '[UNK]'):>14}  {p:.3f}")

<div dir="rtl">

## ٦. الخلاصة والخطوة الجاية · Recap & handoff

- ضفنا **كتلة انتباه واحدة برأس واحد** فوق التضمينات.
- حسبنا نمط الانتباه **بإيدينا** وأثبتنا إنه نفس اللي الموديل بيعمله (الفرق صفر) — يعني الخريطة الحرارية حقيقية مش مجاز.
- القدرة الجديدة: الكلمة بقت تقدر **تبص للي قبلها**؛ التوقع بقى يعتمد على السياق.

**المرحلة الجاية (٢ب):** نضيف **كذا رأس** (multiple heads) في نفس الطبقة، ونشوف كل رأس بيتخصص في علاقة مختلفة — يعني الموديل يمسك أكتر من علاقة في نفس الوقت. ده ممكن دلوقتي لأننا فهمنا رأس واحد لوحده.

</div>

<div dir="ltr">

## 6. Recap & handoff

- We added **one attention block with one head** on top of the embeddings.
- We computed the attention pattern **by hand** and proved it equals the model's own (diff ≈ 0) — the heatmap is the matrix, not a metaphor for it.
- New capability: a token can now **look at the tokens before it**; the prediction became context-dependent.

**Next (2b):** add **multiple heads** in the same layer and watch each head specialise on a different Arabic relation — several relations captured at once. We can do this now that we understand a single head on its own.

</div>